# Unemployment and crime in Polish voivodeships: data cleaning

In this notebook I clean the data downloaded from the GUS Local Data Bank (BDL)
with `scripts/fetch_data.py` and prepare two tables for the analysis and the dashboard:
- **unemployment**: unemployment rate and population for each voivodeship and year
- **crimes**: number of crimes and crimes per 1,000 inhabitants for each voivodeship, year and crime category

## 1. First look at the raw data

Let's check what we downloaded: column types, missing values and the range of years.

In [1]:
import pandas as pd
from pathlib import Path

read_path= Path("../data/raw")
df_unp = pd.read_csv(read_path/"bdl_data_unemployment.csv",dtype={"unit_id":str})

print(df_unp.head())
print(df_unp.info())
print(df_unp.isna().sum())

        unit_id  voivodeship  year  value           variable
0  011200000000  MAŁOPOLSKIE  2004   15.0  unemployment_rate
1  011200000000  MAŁOPOLSKIE  2005   13.8  unemployment_rate
2  011200000000  MAŁOPOLSKIE  2006   11.3  unemployment_rate
3  011200000000  MAŁOPOLSKIE  2007    8.7  unemployment_rate
4  011200000000  MAŁOPOLSKIE  2008    7.5  unemployment_rate
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 848 entries, 0 to 847
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   unit_id      848 non-null    object 
 1   voivodeship  848 non-null    object 
 2   year         848 non-null    int64  
 3   value        848 non-null    float64
 4   variable     848 non-null    object 
dtypes: float64(1), int64(1), object(3)
memory usage: 33.2+ KB
None
unit_id        0
voivodeship    0
year           0
value          0
variable       0
dtype: int64


In [2]:
df_crime = pd.read_csv(read_path/"bdl_data_crime.csv",dtype={"unit_id":str})

print(df_crime.head())
print(df_crime.info())
print(df_crime.isna().sum())

        unit_id  voivodeship  year     value      variable
0  011200000000  MAŁOPOLSKIE  2002  106208.0  crimes_total
1  011200000000  MAŁOPOLSKIE  2003  129312.0  crimes_total
2  011200000000  MAŁOPOLSKIE  2004  126953.0  crimes_total
3  011200000000  MAŁOPOLSKIE  2005  115179.0  crimes_total
4  011200000000  MAŁOPOLSKIE  2006  105630.0  crimes_total
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4416 entries, 0 to 4415
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   unit_id      4416 non-null   object 
 1   voivodeship  4416 non-null   object 
 2   year         4416 non-null   int64  
 3   value        4416 non-null   float64
 4   variable     4416 non-null   object 
dtypes: float64(1), int64(1), object(3)
memory usage: 172.6+ KB
None
unit_id        0
voivodeship    0
year           0
value          0
variable       0
dtype: int64


There are no missing values in both tables.
`unit_id` must be read as text (`dtype={"unit_id": str}`). Otherwise pandas reads it as a number
and removes the leading zero (`011200000000` becomes `11200000000`), so it would not match other data later.

In [3]:
print(df_unp.groupby("variable")["year"].agg(["min","max","count"]))

print(df_crime.groupby("variable")["year"].agg(["min","max","count"]))

                    min   max  count
variable                            
population         1995  2025    496
unemployment_rate  2004  2025    352
                              min   max  count
variable                                      
crimes_criminal              2002  2024    368
crimes_criminal_per_1000     2002  2024    368
crimes_economic              2002  2024    368
crimes_economic_per_1000     2002  2024    368
crimes_life_health           2002  2024    368
crimes_life_health_per_1000  2002  2024    368
crimes_property              2002  2024    368
crimes_property_per_1000     2002  2024    368
crimes_road                  2002  2024    368
crimes_road_per_1000         2002  2024    368
crimes_total                 2002  2024    368
crimes_total_per_1000        2002  2024    368


All variables are complete: every voivodeship has a value for every year.
The years are different for each source:
- population: 1995-2025
- unemployment rate: 2004-2025
- crimes: 2002-2024

The years that all variables have in common are 2004-2024.

## 2. Choosing the years

I use only the years **2014-2024**, because the crime data is not fully comparable before and after:
- **2013**: crimes from investigations led by the prosecutor's office and other bodies are no longer counted
- **2014**: crimes committed by minors are no longer counted
- **2025**: the police changed the crime categories


### Checking the method change in 2014

How much did crime per 1,000 inhabitants change from 2013 to 2014 in each voivodeship?

In [4]:
df_check = df_crime[(df_crime["variable"]=="crimes_total_per_1000") & (df_crime["year"].isin([2012,2013,2014,2015]))]
df_check=df_check.pivot(index="voivodeship", columns="year",values="value")
df_check["change 2012-2013"] = (df_check[2013] - df_check[2012])/df_check[2012]*100
df_check["change 2013-2014"] = (df_check[2014] - df_check[2013])/df_check[2013]*100
df_check["change 2014-2015"] = (df_check[2015] - df_check[2014]) / df_check[2014]*100
df_check

year,2012,2013,2014,2015,change 2012-2013,change 2013-2014,change 2014-2015
voivodeship,,,,,,,
DOLNOŚLĄSKIE,39.49,38.15,28.73,27.33,-3.393264,-24.692005,-4.872955
KUJAWSKO-POMORSKIE,28.96,25.53,20.12,18.60,-11.843923,-21.190756,-7.554672
LUBELSKIE,22.56,20.68,17.14,15.39,-8.333333,-17.117988,-10.210035
LUBUSKIE,37.82,37.41,29.13,26.43,-1.084082,-22.133119,-9.268795
MAZOWIECKIE,26.99,26.31,22.26,21.06,-2.519452,-15.393387,-5.390836
MAŁOPOLSKIE,28.93,25.78,21.90,20.56,-10.888351,-15.050427,-6.118721
OPOLSKIE,29.55,28.45,22.92,19.95,-3.722504,-19.437610,-12.958115
PODKARPACKIE,17.92,16.76,13.11,12.92,-6.473214,-21.778043,-1.449275
PODLASKIE,20.95,18.69,15.51,14.68,-10.787589,-17.014446,-5.351386


In most voivodeships crime per 1,000 inhabitants was going down every year,
usually by 0-12% (2012-2013) and 1-13% (2014-2015).
From 2013 to 2014 the drop was much bigger, mostly 15-22%.
Part of it is probably caused by the change in the method (minors are not counted from 2014).
Because the data before and after 2014 is counted differently, I only use 2014-2024.

In [5]:
crime = df_crime[df_crime["year"].between(2014,2024)].copy()
unp = df_unp[df_unp["year"].between(2014,2024)].copy()

print(crime.groupby("variable")["year"].agg(["min", "max", "count"]))
print(unp.groupby("variable")["year"].agg(["min", "max", "count"]))

                              min   max  count
variable                                      
crimes_criminal              2014  2024    176
crimes_criminal_per_1000     2014  2024    176
crimes_economic              2014  2024    176
crimes_economic_per_1000     2014  2024    176
crimes_life_health           2014  2024    176
crimes_life_health_per_1000  2014  2024    176
crimes_property              2014  2024    176
crimes_property_per_1000     2014  2024    176
crimes_road                  2014  2024    176
crimes_road_per_1000         2014  2024    176
crimes_total                 2014  2024    176
crimes_total_per_1000        2014  2024    176
                    min   max  count
variable                            
population         2014  2024    176
unemployment_rate  2014  2024    176


After filtering, every variable has the years 2014-2024 and 176 rows (11 years x 16 voivodeships).

## 3. Final tables

**unemployment**: one row for each voivodeship and year, with a column for each variable.



In [6]:
unemployment = unp.pivot(
    index = ["unit_id", "voivodeship", "year"],
    columns ="variable",
    values="value"
).reset_index()
unemployment.columns.name = None
print(unemployment.shape)
unemployment.head()

(176, 5)


,unit_id,voivodeship,year,population,unemployment_rate
0,011200000000,MAŁOPOLSKIE,2014,3368336.0,9.7
1,011200000000,MAŁOPOLSKIE,2015,3372618.0,8.3
2,011200000000,MAŁOPOLSKIE,2016,3382260.0,6.6
3,011200000000,MAŁOPOLSKIE,2017,3391380.0,5.3
4,011200000000,MAŁOPOLSKIE,2018,3400577.0,4.7


In [7]:
unemployment["population"] = unemployment["population"].astype(int)
unemployment.head()

,unit_id,voivodeship,year,population,unemployment_rate
0,011200000000,MAŁOPOLSKIE,2014,3368336,9.7
1,011200000000,MAŁOPOLSKIE,2015,3372618,8.3
2,011200000000,MAŁOPOLSKIE,2016,3382260,6.6
3,011200000000,MAŁOPOLSKIE,2017,3391380,5.3
4,011200000000,MAŁOPOLSKIE,2018,3400577,4.7


The unemployment table has 176 rows: 16 voivodeships x 11 years.
Population is changed to a whole number (it was a decimal, because in the raw data
the `value` column had both percentages and population).

**crimes**: one row for each voivodeship, year and crime category,
with two columns: number of crimes and crimes per 1,000 inhabitants.


In [8]:
crime["variable"].unique()

array(['crimes_total', 'crimes_criminal', 'crimes_economic',
       'crimes_road', 'crimes_property', 'crimes_life_health',
       'crimes_total_per_1000', 'crimes_criminal_per_1000',
       'crimes_economic_per_1000', 'crimes_road_per_1000',
       'crimes_property_per_1000', 'crimes_life_health_per_1000'],
      dtype=object)

In [9]:
import numpy as np
condition = crime["variable"].str.endswith("_per_1000")
crime["measure"]=np.where(condition,"crimes_per_1000","crime_count")
crime.head()

,unit_id,voivodeship,year,value,variable,measure
12,011200000000,MAŁOPOLSKIE,2014,73687.0,crimes_total,crime_count
13,011200000000,MAŁOPOLSKIE,2015,69300.0,crimes_total,crime_count
14,011200000000,MAŁOPOLSKIE,2016,72583.0,crimes_total,crime_count
15,011200000000,MAŁOPOLSKIE,2017,82410.0,crimes_total,crime_count
16,011200000000,MAŁOPOLSKIE,2018,74697.0,crimes_total,crime_count


In [10]:
crime["category"]=crime["variable"].str.replace("crimes_","").str.replace("_per_1000","")
crime

,unit_id,voivodeship,year,value,variable,measure,category
12,011200000000,MAŁOPOLSKIE,2014,73687.00,crimes_total,crime_count,total
13,011200000000,MAŁOPOLSKIE,2015,69300.00,crimes_total,crime_count,total
14,011200000000,MAŁOPOLSKIE,2016,72583.00,crimes_total,crime_count,total
15,011200000000,MAŁOPOLSKIE,2017,82410.00,crimes_total,crime_count,total
16,011200000000,MAŁOPOLSKIE,2018,74697.00,crimes_total,crime_count,total
...,...,...,...,...,...,...,...
4411,071400000000,MAZOWIECKIE,2020,0.31,crimes_life_health_per_1000,crimes_per_1000,life_health
4412,071400000000,MAZOWIECKIE,2021,0.30,crimes_life_health_per_1000,crimes_per_1000,life_health
4413,071400000000,MAZOWIECKIE,2022,0.29,crimes_life_health_per_1000,crimes_per_1000,life_health
4414,071400000000,MAZOWIECKIE,2023,0.30,crimes_life_health_per_1000,crimes_per_1000,life_health


In [11]:
crime[["variable", "category", "measure"]].drop_duplicates()

,variable,category,measure
12,crimes_total,total,crime_count
380,crimes_criminal,criminal,crime_count
748,crimes_economic,economic,crime_count
1116,crimes_road,road,crime_count
1484,crimes_property,property,crime_count
1852,crimes_life_health,life_health,crime_count
2220,crimes_total_per_1000,total,crimes_per_1000
2588,crimes_criminal_per_1000,criminal,crimes_per_1000
2956,crimes_economic_per_1000,economic,crimes_per_1000
3324,crimes_road_per_1000,road,crimes_per_1000


Now every row has a `category` (e.g. `road`) and a `measure`
(`crime_count` = number of crimes, `crimes_per_1000` = crimes per 1,000 inhabitants).
I checked the unique combinations: 12 variables -> 6 categories, each with both measures.

In [12]:
crimes = crime.pivot(
    index = ["unit_id", "voivodeship", "year","category"],
    columns ="measure",
    values="value"
).reset_index()
crimes.columns.name = None
print(crimes.shape)
crimes.head()

(1056, 6)


,unit_id,voivodeship,year,category,crime_count,crimes_per_1000
0,011200000000,MAŁOPOLSKIE,2014,criminal,51429.0,15.29
1,011200000000,MAŁOPOLSKIE,2014,economic,13625.0,4.05
2,011200000000,MAŁOPOLSKIE,2014,life_health,1608.0,0.48
3,011200000000,MAŁOPOLSKIE,2014,property,38955.0,11.58
4,011200000000,MAŁOPOLSKIE,2014,road,6309.0,1.88


I pivoted the data so each row is one voivodeship, one year and one crime category,
with two columns: `crime_count` and `crimes_per_1000`.
This way `category` can be used as a filter in later steps.

In [13]:
crimes["crime_count"] = crimes["crime_count"].astype(int)
crimes.head()

,unit_id,voivodeship,year,category,crime_count,crimes_per_1000
0,011200000000,MAŁOPOLSKIE,2014,criminal,51429,15.29
1,011200000000,MAŁOPOLSKIE,2014,economic,13625,4.05
2,011200000000,MAŁOPOLSKIE,2014,life_health,1608,0.48
3,011200000000,MAŁOPOLSKIE,2014,property,38955,11.58
4,011200000000,MAŁOPOLSKIE,2014,road,6309,1.88


`crime_count` became a float after the pivot (because it shared the `value` column
with the per 1,000 rates), so I changed it back to int.

## 4. Checking the GUS rate
I calculated my own rate (`crime_count / population * 1000`) and compared it with `crimes_per_1000` from GUS.




In [14]:
check = crimes.merge(unemployment[["unit_id","year","population"]],on=["unit_id","year"])
check.shape

(1056, 7)

In [15]:
check["my_per_1000"]=check["crime_count"]/check["population"]*1000
check["diff"]=check["my_per_1000"]-check["crimes_per_1000"]
check["diff"].abs().max()

np.float64(0.11506773808279647)

In [16]:
check["diff"].describe()

count    1056.000000
mean        0.010436
std         0.020598
min        -0.054179
25%        -0.000506
50%         0.004967
75%         0.017723
max         0.115068
Name: diff, dtype: float64

In [17]:
check[["crimes_per_1000", "my_per_1000"]].head()

,crimes_per_1000,my_per_1000
0,15.29,15.268370
1,4.05,4.045024
2,0.48,0.477387
3,11.58,11.565058
4,1.88,1.873032


In [18]:
check["diff_pct"] = check["diff"] / check["crimes_per_1000"] * 100
check["diff_pct"].describe()

count    1056.000000
mean        0.129952
std         0.358936
min        -1.447152
25%        -0.016826
50%         0.134555
75%         0.281973
max         2.233422
Name: diff_pct, dtype: float64

In [19]:
check.groupby("year")["diff"].mean()

year
2014    0.003691
2015    0.007434
2016    0.000886
2017   -0.000165
2018    0.003154
2019    0.003819
2020    0.021233
2021    0.022961
2022    0.017831
2023    0.016598
2024    0.017349
Name: diff, dtype: float64

The differences are very small: median 0.13%, all between -1.5% and +2.3%.
The biggest % differences are probably in small categories (like `life_health`), where rounding to 2 decimals matters a lot.
From 2020 the differences are a bit bigger, probably because GUS recalculated population after the 2021 census. The GUS rates match the data, so I keep them as they are.

## 5. Saving
I saved two clean tables to `data/processed/`:
- `unemployment.csv` - one row per voivodeship and year (population, unemployment rate)
- `crimes.csv` - one row per voivodeship, year and crime category (crime count, crimes per 1,000)

In [20]:
save_path = Path("../data/processed/")
unemployment.to_csv(save_path/"unemployment.csv",index=False)
crimes.to_csv(save_path/"crimes.csv",index=False)